In [8]:
import os
import time
import psycopg2
from openai import OpenAI
from dotenv import load_dotenv

# 1. 환경 변수 로드
load_dotenv()

# == ⚙️ 설정 정보 ==
GMS_KEY = os.getenv("GMS_API_KEY")
GMS_BASE_URL = "https://gms.ssafy.io/gmsapi/api.openai.com/v1"
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIM = 1536
BATCH_SIZE = 8

# 저장할 Flyway용 SQL 파일 경로
OUTPUT_SQL_FILE = "V4__seed_category_embeddings.sql"

DB_CONFIG = {
    "host": "localhost",
    "database": "ozazak",
    "user": "b205admin",
    "password": "b205admin", # .env 비밀번호 사용 권장
    "port": "5432"
}

# 2. 클라이언트 설정
if not GMS_KEY:
    raise ValueError("❌ .env 파일에 'GMS_API_KEY'가 없습니다.")

client = OpenAI(api_key=GMS_KEY, base_url=GMS_BASE_URL)

def get_embeddings_batch(texts, model=EMBEDDING_MODEL, dim=EMBEDDING_DIM):
    try:
        clean_texts = [t if t and t.strip() else " " for t in texts]
        resp = client.embeddings.create(
            model=model,
            input=clean_texts,
            dimensions=dim
        )
        return [d.embedding for d in resp.data]
    except Exception as e:
        print(f"❌ API 호출 에러: {e}")
        return None

# 3. 메인 로직
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

# 파일 열기 (쓰기 모드 'w', 인코딩 utf-8)
with open(OUTPUT_SQL_FILE, "w", encoding="utf-8") as sql_file:
    
    # SQL 파일 헤더 작성
    sql_file.write("-- Generated by Python embedding script\n")
    sql_file.write(f"-- Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    try:
        print(f"🔌 DB 연결 성공. 임베딩 작업 및 SQL 파일 저장({OUTPUT_SQL_FILE}) 시작...")

        cur.execute("SELECT id, name FROM category WHERE embedding IS NULL") 
        rows = cur.fetchall()
        
        total_count = len(rows)
        print(f"📊 처리할 데이터: 총 {total_count}건")

        if total_count == 0:
            print("🎉 모든 데이터가 이미 처리되었습니다.")

        for start_idx in range(0, total_count, BATCH_SIZE):
            end_idx = min(start_idx + BATCH_SIZE, total_count)
            batch_rows = rows[start_idx:end_idx]
            
            batch_ids = [row[0] for row in batch_rows]
            batch_texts = [row[1] for row in batch_rows]

            vectors = get_embeddings_batch(batch_texts)

            if vectors:
                for i, vector in enumerate(vectors):
                    cat_id = batch_ids[i]
                    
                    # 1️⃣ DB 실시간 업데이트
                    cur.execute(
                        "UPDATE category SET embedding = %s WHERE id = %s",
                        (vector, cat_id)
                    )
                    
                    # 2️⃣ SQL 파일에 쿼리 저장
                    # Python 리스트 str(vector)는 '[0.1, 0.2 ...]' 형태가 되어 PostgreSQL 문법과 호환됩니다.
                    # 안전을 위해 ::vector 형변환을 명시합니다.
                    sql_query = f"UPDATE category SET embedding = '{str(vector)}'::vector WHERE id = {cat_id};\n"
                    sql_file.write(sql_query)

                conn.commit()
                # 파일 버퍼 비우기 (중간에 꺼져도 파일에 남도록)
                sql_file.flush() 
                
                print(f"✅ [{end_idx}/{total_count}] DB 업데이트 & 파일 기록 완료")
            
            time.sleep(0.1)

        print(f"🏁 작업 완료! '{OUTPUT_SQL_FILE}' 파일이 생성되었습니다.")

    except Exception as e:
        conn.rollback()
        print(f"❌ 작업 중 에러: {e}")
    finally:
        cur.close()
        conn.close()

🔌 DB 연결 성공. 임베딩 작업 및 SQL 파일 저장(V4__seed_category_embeddings.sql) 시작...
📊 처리할 데이터: 총 241건
✅ [8/241] DB 업데이트 & 파일 기록 완료
✅ [16/241] DB 업데이트 & 파일 기록 완료
✅ [24/241] DB 업데이트 & 파일 기록 완료
✅ [32/241] DB 업데이트 & 파일 기록 완료
✅ [40/241] DB 업데이트 & 파일 기록 완료
✅ [48/241] DB 업데이트 & 파일 기록 완료
✅ [56/241] DB 업데이트 & 파일 기록 완료
✅ [64/241] DB 업데이트 & 파일 기록 완료
✅ [72/241] DB 업데이트 & 파일 기록 완료
✅ [80/241] DB 업데이트 & 파일 기록 완료
✅ [88/241] DB 업데이트 & 파일 기록 완료
✅ [96/241] DB 업데이트 & 파일 기록 완료
✅ [104/241] DB 업데이트 & 파일 기록 완료
✅ [112/241] DB 업데이트 & 파일 기록 완료
✅ [120/241] DB 업데이트 & 파일 기록 완료
✅ [128/241] DB 업데이트 & 파일 기록 완료
✅ [136/241] DB 업데이트 & 파일 기록 완료
✅ [144/241] DB 업데이트 & 파일 기록 완료
✅ [152/241] DB 업데이트 & 파일 기록 완료
✅ [160/241] DB 업데이트 & 파일 기록 완료
✅ [168/241] DB 업데이트 & 파일 기록 완료
✅ [176/241] DB 업데이트 & 파일 기록 완료
✅ [184/241] DB 업데이트 & 파일 기록 완료
✅ [192/241] DB 업데이트 & 파일 기록 완료
✅ [200/241] DB 업데이트 & 파일 기록 완료
✅ [208/241] DB 업데이트 & 파일 기록 완료
✅ [216/241] DB 업데이트 & 파일 기록 완료
✅ [224/241] DB 업데이트 & 파일 기록 완료
✅ [232/241] DB 업데이트 & 파일 기록 완료
✅ [240/241] DB 업데이트 & 파일 